In [5]:
import pandas as pd
import wget
from zipfile import ZipFile
import os

def download_extract_concatenate_dfp_files(start_year, end_year, financial_statements):
    """
    Função para realizar o download, extração e concatenação de arquivos zipados contendo os dados da Demonstração Financeira
    Padronizada (DFP) de companhias abertas disponibilizados pela Comissão de Valores Mobiliários (CVM).
    """
    url = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/"
    
    # Cria uma lista vazia para inserir os nomes dos arquivos zipados
    arquivo_zipado = []
    
    # Define os nomes dos arquivos zipados de acordo com o range de datas
    for ano in range(start_year, end_year + 1):
        arquivo_zipado.append(f'dfp_cia_aberta_{ano}.zip')
        
    # Realiza o download dos arquivos zipados de acordo com a url base
    for arquivos in arquivo_zipado:
        wget.download(url + arquivos)
        
    # Extrai os arquivos zipados
    for arquivos in arquivo_zipado:
        ZipFile(arquivos, 'r').extractall('DFP')
        
    # Concatena os dados dos demonstrativos financeiros em um único DataFrame
    for demons in financial_statements:
        arquivo_demonstrativo = pd.DataFrame()
        for ano in range(start_year, end_year + 1):
            arquivo_ano = pd.read_csv(f'DFP/dfp_cia_aberta_{demons}_{ano}.csv', sep=';', decimal=',', encoding='ISO-8859-1')
            arquivo_demonstrativo = pd.concat([arquivo_demonstrativo, arquivo_ano])
            
        arquivo_demonstrativo.to_csv(f'dfp_cia_aberta_{demons}_{start_year}-{end_year}.csv', index=False)

In [6]:
# Coleta os dados da DRE (ajustado para o período das imagens)
download_extract_concatenate_dfp_files(2021, 2023, ['DRE_con'])

# Realiza a leitura do arquivo gerado
dre = pd.read_csv('dfp_cia_aberta_DRE_con_2021-2023.csv')

# Realiza o filtro das contas contábeis, do código CVM e da ordem do exercício
dados = dre[
    dre.CD_CONTA.isin([
        "3.01", # Receita de Venda de Bens e/ou Serviços
        "3.03", # Resultado Bruto
        "3.05", # EBIT
        "3.11"  # Lucro/Prejuízo Consolidado do Período
    ]) & 
    dre.CD_CVM.isin([12190, 26700, 21431, 19550]) & 
    (dre.ORDEM_EXERC == 'ÚLTIMO')
]

# Seleciona as colunas de interesse
dados = dados[["DT_REFER", "DENOM_CIA", "CD_CONTA", "DS_CONTA", "VL_CONTA"]]

In [7]:
# Converte a coluna de data para datetime
dados['DT_REFER'] = pd.to_datetime(dados['DT_REFER'])

# Cria a coluna Ano
dados['Ano'] = dados['DT_REFER'].dt.year

# Pivotagem dos dados para colocar as contas como colunas
dados_pivot = dados.pivot(index=['Ano', 'DENOM_CIA'], columns='CD_CONTA', values='VL_CONTA')

# Cria os indicadores de margem
indicadores = (
    dados_pivot
    .assign(
        margem_bruta = (dados_pivot["3.03"]) / dados_pivot["3.01"] * 100,
        margem_liquida = (dados_pivot["3.11"]) / dados_pivot["3.01"] * 100,
        margem_ebit = (dados_pivot["3.05"]) / dados_pivot["3.01"] * 100
    )
)

# Retira o índice e seleciona/arredonda as colunas
indicadores.reset_index(inplace=True)
indicadores = indicadores[['Ano', 'DENOM_CIA', 'margem_bruta', 'margem_liquida', 'margem_ebit']].round(decimals=3)

# Transformação para formato "long" para facilitar a plotagem
indicadores_long = indicadores.melt(id_vars=['Ano', 'DENOM_CIA'], var_name="Indicadores")

In [11]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

colors = {
    'margem_liquida': '#282f6b',
    'margem_bruta':   '#b22200',
    'margem_ebit':    '#eace3f',
}
labels = {
    'margem_liquida': 'Margem Líquida',
    'margem_bruta':   'Margem Bruta',
    'margem_ebit':    'Margem EBIT',
}

empresas = indicadores_long['DENOM_CIA'].unique().tolist()
n = len(empresas)
ncols = 3
nrows = -(-n // ncols)  # teto sem math.ceil

fig = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=empresas,
    shared_xaxes=False,
    shared_yaxes=False,
    horizontal_spacing=0.08,
    vertical_spacing=0.12,
)

added_to_legend = set()

for i, empresa in enumerate(empresas):
    row = i // ncols + 1
    col = i % ncols + 1
    df_emp = indicadores_long[indicadores_long['DENOM_CIA'] == empresa]

    for indicador in df_emp['Indicadores'].unique():
        df_ind = df_emp[df_emp['Indicadores'] == indicador]
        show_legend = indicador not in added_to_legend

        fig.add_trace(
            go.Scatter(
                x=df_ind['Ano'],
                y=df_ind['value'],
                mode='lines',
                name=labels.get(indicador, indicador),
                line=dict(color=colors.get(indicador, '#666666'), width=2),
                legendgroup=indicador,
                showlegend=show_legend,
            ),
            row=row, col=col,
        )
        added_to_legend.add(indicador)

fig.update_layout(
    title=dict(
        text='Indicadores de Eficiência de empresas do setor Farmacêutico e Higiene',
        font=dict(size=14),
        x=0.5,
        xanchor='center',
    ),
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.08,
        xanchor='center',
        x=0.5,
    ),
    height=350 * nrows,
    width=1100,
    annotations=[
        *fig.layout.annotations,  # mantém os subplot_titles
        dict(
            text='Fonte: CVM/B3',
            xref='paper', yref='paper',
            x=1, y=-0.05,
            xanchor='right', yanchor='top',
            showarrow=False,
            font=dict(size=11, color='#666666'),
        )
    ],
)

fig.update_yaxes(title_text='%')

fig.show()